# Notebook 6: Sensitivity Analysis & Validity of Assumptions## When do these models work? Parameter ranges and breakdown conditions.This notebook systematically explores the parameter space where the theoretical models used in dv/v interpretation remain valid, and identifies conditions where assumptions break down.### ReferencesAll project papers: Okubo et al. (2024), Clements & Denolle (2023), Ermert et al. (2023), Richter et al. (2014), Fokker et al. (2021), Tromp & Trampert (2018), Murnaghan (1937).

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom matplotlib.patches import Rectanglefrom matplotlib.collections import PatchCollectionplt.rcParams.update({    'font.size': 12, 'axes.labelsize': 14, 'figure.dpi': 150,    'font.family': 'serif', 'mathtext.fontset': 'cm',    'axes.grid': True, 'grid.alpha': 0.3})print("Environment ready.")

## 1. Assumption: Homogeneous Half-SpaceThe Berger (1975) thermoelastic and Roeloffs (1988) poroelastic solutions assume a **homogeneous half-space**. Real geology is layered, and the dv/v measurementis depth-averaged by surface-wave sensitivity kernels.**When does this break down?**- When velocity contrasts are strong (e.g., soft sediment over bedrock)- When measurement frequency implies sensitivity deeper than the homogeneous layer- Ermert et al. (2023) use station-specific velocity profiles and sensitivity kernels

In [ ]:
# === Sensitivity Kernel vs Homogeneous Assumption ===def rayleigh_sensitivity_kernel(z, f, Vs, mode='fundamental'):    """    Approximate Rayleigh wave phase velocity sensitivity kernel.    Peak sensitivity at ~λ/3 depth.    """    wavelength = Vs / f    z_peak = wavelength / 3    # Gaussian approximation    sigma = wavelength / 4    kernel = np.exp(-0.5 * ((z - z_peak) / sigma)**2)    kernel /= kernel.max()    return kernelz = np.linspace(0, 2000, 500)frequencies = [0.3, 0.5, 1.0, 2.0, 4.0]Vs = 600  # m/s (near-surface)fig, axes = plt.subplots(1, 3, figsize=(16, 7))# Panel (a): Sensitivity kernels for different frequenciesax = axes[0]for f in frequencies:    K = rayleigh_sensitivity_kernel(z, f, Vs)    ax.plot(K, z, lw=2, label=f'{f} Hz')ax.set_xlabel('Sensitivity [normalized]')ax.set_ylabel('Depth [m]')ax.set_title(f'(a) Rayleigh wave sensitivity kernels\n(Vs = {Vs} m/s)')ax.invert_yaxis()ax.legend()# Shade regions where homogeneous assumption failsfor layer_depth in [100, 300]:    ax.axhline(layer_depth, color='red', ls='--', alpha=0.5)ax.text(0.5, 105, 'Layer 1 boundary', color='red', fontsize=9)# Panel (b): Effective depth of measurement vs frequencyax = axes[1]freq_range = np.logspace(-1, 1, 100)Vs_values = [200, 400, 800, 1500, 3000]for Vs_val in Vs_values:    depth_peak = Vs_val / freq_range / 3    ax.loglog(freq_range, depth_peak, lw=2, label=f'Vs = {Vs_val} m/s')ax.set_xlabel('Frequency [Hz]')ax.set_ylabel('Peak sensitivity depth [m]')ax.set_title('(b) Measurement depth vs frequency')ax.legend(fontsize=9)ax.axhspan(0, 10, alpha=0.1, color='green', label='Thermoelastic domain')ax.axhspan(10, 500, alpha=0.1, color='blue')# Panel (c): When does layering matter?ax = axes[2]# Compute dv/v error from using homogeneous vs layered modelvelocity_contrast = np.linspace(1.0, 5.0, 100)  # Vs ratio across boundarylayer_fraction = np.linspace(0, 1, 100)  # fraction of kernel in wrong layerVC, LF = np.meshgrid(velocity_contrast, layer_fraction)# Error ~ velocity contrast * fraction of energy in wrong layererror_percent = (VC - 1) * LF * 50  # rough estimateim = ax.contourf(velocity_contrast, layer_fraction, error_percent,                  levels=np.arange(0, 105, 5), cmap='YlOrRd')plt.colorbar(im, ax=ax, label='Relative error in dv/v [%]')ax.set_xlabel('Velocity contrast (Vs2/Vs1)')ax.set_ylabel('Fraction of sensitivity in deeper layer')ax.set_title('(c) Error from homogeneous assumption')ax.contour(velocity_contrast, layer_fraction, error_percent, levels=[10, 25, 50],            colors='k', linewidths=1)fig.suptitle('Validity of Homogeneous Half-Space Assumption', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig15_homogeneous_validity.png', bbox_inches='tight')plt.show()

## 2. Assumption: Linear Nonlinear Elasticity ($dv/v = \beta\epsilon_{kk}$)The acoustoelastic relation $dv/v = \beta\epsilon_{kk}$ assumes:1. Strain is small enough for the first-order expansion to hold2. $\beta$ is constant (no strain-dependent $\beta$)3. No hysteretic or mesoscopic nonlinearity**Breakdown conditions:**- Strains > $10^{-5}$: higher-order terms become significant- Near damage threshold: cracks nucleate/coalesce- Dynamic loading: rate-dependent effects (slow dynamics)

In [ ]:
# === Validity of Linear Acoustoelastic Approximation ===def dvv_exact_nonlinear(epsilon, beta, gamma=0, delta=0):    """    dv/v including higher-order terms:    dv/v = beta*eps + gamma*eps^2 + delta*eps^3    """    return beta * epsilon + gamma * epsilon**2 + delta * epsilon**3epsilon_range = np.logspace(-9, -3, 1000)fig, axes = plt.subplots(1, 2, figsize=(14, 6))# Panel (a): dv/v for different nonlinear ordersax = axes[0]beta = -500gamma_vals = [0, 1e5, 1e7, 1e9]for gamma in gamma_vals:    dvv_pos = dvv_exact_nonlinear(epsilon_range, beta, gamma)    dvv_neg = dvv_exact_nonlinear(-epsilon_range, beta, gamma)    ax.loglog(epsilon_range, np.abs(dvv_pos), lw=2,               label=f'$\\gamma$ = {gamma:.0e}' if gamma > 0 else 'Linear only')ax.axvline(1e-6, color='blue', ls=':', alpha=0.7, label='Tidal strain')ax.axvline(1e-5, color='green', ls=':', alpha=0.7, label='Thermoelastic')ax.axvline(1e-4, color='red', ls=':', alpha=0.7, label='Near-earthquake')ax.set_xlabel('Strain $|\\epsilon|$')ax.set_ylabel('|dv/v|')ax.set_title('(a) Linearity breakdown: strain thresholds')ax.legend(fontsize=9)# Panel (b): Relative error from linear approximationax = axes[1]for gamma in [1e5, 1e7, 1e9]:    dvv_full = dvv_exact_nonlinear(epsilon_range, beta, gamma)    dvv_linear = beta * epsilon_range    rel_error = np.abs((dvv_full - dvv_linear) / dvv_linear) * 100    ax.loglog(epsilon_range, rel_error, lw=2, label=f'$\\gamma$ = {gamma:.0e}')ax.axhline(1, color='k', ls='--', alpha=0.5, label='1% error')ax.axhline(10, color='k', ls=':', alpha=0.5, label='10% error')ax.axvline(1e-6, color='blue', ls=':', alpha=0.7)ax.axvline(1e-5, color='green', ls=':', alpha=0.7)ax.set_xlabel('Strain $|\\epsilon|$')ax.set_ylabel('Relative error from linear approx. [%]')ax.set_title('(b) When does linear approximation fail?')ax.legend(fontsize=9)ax.set_ylim(1e-4, 1e4)fig.suptitle('Validity Range of Linear Acoustoelastic Approximation', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig16_linearity_validity.png', bbox_inches='tight')plt.show()print("Summary of strain regimes:")print("  Tidal:          ~50 nanostrain  → linear approximation excellent")print("  Thermoelastic:  ~1-10 microstrain → linear usually valid")print("  Hydrological:   ~1-50 microstrain → depends on material")print("  Coseismic:      ~100+ microstrain → nonlinear effects important")print("  Strong shaking: ~1000+ microstrain → damage, slow dynamics")

## 3. Comprehensive Parameter MapHere we synthesize the key parameters and their typical ranges across differentgeological settings and applications.

In [ ]:
# === Comprehensive Parameter Overview ===fig, ax = plt.subplots(figsize=(16, 10))ax.set_xlim(0, 10)ax.set_ylim(0, 10)ax.axis('off')# Titleax.text(5, 9.7, 'Parameter Ranges & Validity Conditions for dv/v Models',        ha='center', fontsize=16, fontweight='bold')# Table contentheaders = ['Parameter', 'Symbol', 'Typical Range', 'Critical For', 'Validity Limit']data = [    ['Thermal diffusivity', '$\\kappa_T$', '0.15–2.0 mm²/s', 'Thermoelastic', 'Layered media'],    ['Hydraulic diffusivity', 'c', '0.001–10 m²/s', 'Hydrological', 'Heterogeneous k'],    ['Poissons ratio', '$\\nu$', '0.15–0.35', 'Stress partitioning', 'Anisotropy'],    ['Undrained $\\nu$', '$\\nu_u$', '0.3–0.5', 'Poroelastic', 'Partial saturation'],    ['Skempton coeff.', 'B', '0.2–1.0', 'Pore pressure', 'Depth-dependent'],    ['Acoustoelastic $\\beta$', '$\\beta$', '−10 to −10⁴', 'dv/v amplitude', 'ε > 10⁻⁵'],    ['TOE constants', 'l, m, n', '−10¹⁰ to −10¹²', 'Nonlinear response', 'Material-specific'],    ['$\\partial(\\rho v^2)/\\partial\\sigma_c$', "$\\mu'$", '5–1000', 'Stress sensitivity', 'Confining P'],    ['Thermal expansion', '$\\alpha$', '5–15 × 10⁻⁶ K⁻¹', 'Thermoelastic', 'Temperature range'],    ['Porosity', '$\\phi$', '0.01–0.40', 'Pore pressure', 'Dual porosity'],    ['Crack density', '$\\epsilon_c$', '0–0.2', 'Anisotropy', 'Percolation'],    ['Healing $\\tau_{min}$', '$\\tau_{min}$', '0.01–5 yr', 'Post-seismic', 'Slow dynamics'],    ['Healing $\\tau_{max}$', '$\\tau_{max}$', '1–30,000 yr', 'Long-term trend', 'Data duration'],]y_start = 9.0row_height = 0.52col_positions = [0.2, 2.5, 4.3, 6.5, 8.3]# Headersfor j, (header, xpos) in enumerate(zip(headers, col_positions)):    ax.text(xpos, y_start, header, fontsize=11, fontweight='bold',            va='center')# Separator lineax.plot([0.1, 9.9], [y_start - 0.25, y_start - 0.25], 'k-', lw=1)# Data rowsfor i, row in enumerate(data):    y = y_start - 0.25 - (i + 1) * row_height    bg_color = '#f0f0f0' if i % 2 == 0 else 'white'    rect = plt.Rectangle((0.1, y - 0.2), 9.8, row_height,                           facecolor=bg_color, edgecolor='none')    ax.add_patch(rect)    for j, (val, xpos) in enumerate(zip(row, col_positions)):        ax.text(xpos, y, val, fontsize=10, va='center')plt.tight_layout()plt.savefig('fig17_parameter_table.png', bbox_inches='tight')plt.show()

## 4. Application Scenarios: Where Does Each Model Dominate?

In [ ]:
# === Regime diagram ===fig, ax = plt.subplots(figsize=(14, 8))# Create regime diagram: Frequency vs Depthfreq = np.logspace(-1.5, 1.5, 100)depth = np.logspace(-1, 4, 100)F, D = np.meshgrid(freq, depth)# Define rough boundariesax.fill_between([0.03, 0.3], 0.1, 10, alpha=0.2, color='red', label='Tidal/tectonic')ax.fill_between([0.3, 3], 0.1, 50, alpha=0.2, color='blue', label='Thermoelastic')ax.fill_between([0.1, 2], 10, 1000, alpha=0.2, color='green', label='Hydrological')ax.fill_between([0.5, 10], 0.1, 500, alpha=0.2, color='orange', label='Coseismic damage')ax.fill_between([0.01, 0.1], 100, 10000, alpha=0.2, color='purple', label='Ice/surface loading')ax.fill_between([1, 30], 0.1, 5, alpha=0.2, color='brown', label='Daily thermal')# Add labelsax.text(0.08, 2, 'Tidal\nstrain', fontsize=11, ha='center', fontweight='bold')ax.text(1.0, 5, 'Thermoelastic\n(annual)', fontsize=11, ha='center', fontweight='bold')ax.text(0.5, 100, 'Hydrological\n(pore pressure)', fontsize=11, ha='center', fontweight='bold')ax.text(2.0, 50, 'Coseismic\ndamage', fontsize=11, ha='center', fontweight='bold')ax.text(0.03, 1000, 'Ice/snow\nloading', fontsize=11, ha='center', fontweight='bold')ax.text(5, 1, 'Daily\nthermal', fontsize=11, ha='center', fontweight='bold')ax.set_xscale('log')ax.set_yscale('log')ax.set_xlabel('Measurement frequency [Hz]', fontsize=14)ax.set_ylabel('Depth sensitivity [m]', fontsize=14)ax.set_title('Regime Diagram: Which Process Dominates dv/v?\n'             '(Synthesized from project papers)', fontsize=14)ax.legend(loc='upper right', fontsize=10)ax.set_xlim(0.01, 30)ax.set_ylim(0.1, 10000)plt.tight_layout()plt.savefig('fig18_regime_diagram.png', bbox_inches='tight')plt.show()print("\n=== Summary of Model Applicability ===")print("Thermoelastic: Best at f > 0.5 Hz, shallow (<10 m), arid environments")print("Hydrological:  Best at f = 0.1-2 Hz, 10-500 m depth, wet climates")  print("Tectonic:      Best at f < 0.5 Hz, deep, near active faults")print("Ice loading:   Best at f < 0.1 Hz, very deep, glaciated regions")print("Coseismic:     All frequencies, near-fault, transient signal")

## SummaryThis notebook suite has developed the theoretical and computational framework for interpretingdv/v observations in terms of environmental, tectonic, and volcanic processes. Key takeaways:1. **Thermoelastic effects** dominate at shallow depths in arid environments, with annual dv/v   amplitudes of 0.01–0.3% depending on material properties.2. **Hydrological loading** produces competing effects: surface loading increases velocity while    pore pressure decreases it. The balance depends on permeability and drainage.3. **Nonlinear elasticity** (Murnaghan framework) provides the physical link between strain and    velocity change through the acoustoelastic parameter β.4. **Stress-induced anisotropy** explains why dv/v correlates with contractional strain but not   dilatation — oriented microcracks respond to deviatoric stress.5. **Rheological diagnostics** from dv/v–strain crossplots can distinguish elastic, viscoelastic,   and slow-dynamics behavior.6. **Validity ranges** must be carefully assessed: homogeneous half-space assumptions fail for    strongly layered media, and linear acoustoelasticity breaks down at strains above ~10⁻⁵.